## 1. Parámetros y snapshots
Se seleccionan los mismos Run IDs auditados en Silver. Las salidas incluyen un ID determinista del snapshot.
Repetir la misma carga actualiza sus indicadores mediante MERGE, sin agregar otro snapshot identico.

In [0]:
import hashlib
import json
import re
from functools import reduce
from delta.tables import DeltaTable
from pyspark.sql import functions as F

dbutils.widgets.text("run_procesos", "6717455f-416d-4457-9a88-600732d976d6")
dbutils.widgets.text("run_idartes", "95aa87aa-90ee-4104-8f38-d18edc3287b8")
dbutils.widgets.text("run_idipron", "90a6911a-40da-486b-b7c4-9ac9ab37ecfa")
dbutils.widgets.text("ambiente", "dev", "Ambiente: dev o prod")
dbutils.widgets.text("catalogo", "project_dev", "Catalogo de Unity Catalog")
dbutils.widgets.text("storage_account", "addb01", "Storage account del ambiente")

## 2. Lectura y barreras de calidad
Deben existir 6884 contratos, 6518 variantes de procesos y 6447 portafolios en el snapshot seleccionado.
Las claves de contrato deben ser únicas y las fechas de firma/importes validos. No se descartan filas para forzar un resultado.
Los contratos sin proceso permanecen en todos los indicadores contractuales.

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
ambiente = dbutils.widgets.get("ambiente").strip().lower()
catalogo = dbutils.widgets.get("catalogo").strip()
storage_account = dbutils.widgets.get("storage_account").strip()

if ambiente not in {"dev", "prod"}:
    raise ValueError("ambiente debe ser dev o prod")

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalogo):
    raise ValueError("Nombre de catalogo invalido")

if not re.fullmatch(r"[a-z0-9]{3,24}", storage_account):
    raise ValueError("Nombre de storage account invalido")

if ambiente == "prod" and (
    catalogo.lower() == "project_dev" or storage_account == "addb01"
):
    raise ValueError(
        "Para prod debes indicar su catalogo y storage account"
    )

raiz_destino = f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"

print(f"Ambiente: {ambiente}")
print(f"Catalogo: {catalogo}")
print(f"Raiz de almacenamiento esperada: {raiz_destino}")
esquema = f"{catalogo}.gold"
spark.sql(f"DESCRIBE SCHEMA EXTENDED {esquema}").collect()

def leer_uuid(nombre):
    valor = dbutils.widgets.get(nombre).strip()
    if not re.fullmatch(r"[0-9a-fA-F]{8}(?:-[0-9a-fA-F]{4}){3}-[0-9a-fA-F]{12}", valor):
        raise ValueError(f"UUID invalido: {nombre}")
    return valor

runs = {e: leer_uuid(f"run_{e}") for e in ("idartes", "idipron")}
run_procesos = leer_uuid("run_procesos")
snapshot = hashlib.sha256(json.dumps({"contratos": runs, "procesos": run_procesos}, sort_keys=True).encode()).hexdigest()
filtro_contratos = reduce(lambda a, b: a | b, [
    (F.col("entidad") == e) & (F.col("_run_origen") == run) for e, run in runs.items()])
contratos = spark.table(f"{catalogo}.silver.secop_contratos").filter(
    filtro_contratos & (F.col("_run_procesos") == run_procesos))
procesos = spark.table(f"{catalogo}.silver.secop_procesos").filter(
    F.col("entidad").isin("idartes", "idipron") & (F.col("_run_origen") == run_procesos))
portafolios = spark.table(f"{catalogo}.silver.secop_portafolios").filter(
    F.col("entidad").isin("idartes", "idipron") & (F.col("_run_procesos") == run_procesos))

def exigir_conteos(df, esperados, nombre):
    observado = {r.entidad: r['count'] for r in df.groupBy("entidad").count().collect()}
    if observado != esperados:
        raise ValueError(f"{nombre}: conteo incorrecto {observado}")

exigir_conteos(contratos, {"idartes": 3651, "idipron": 3233}, "Contratos")
exigir_conteos(procesos, {"idartes": 3630, "idipron": 2888}, "Procesos")
exigir_conteos(portafolios, {"idartes": 3588, "idipron": 2859}, "Portafolios")
if contratos.groupBy("entidad", "id_contrato").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Contratos duplicados en el snapshot seleccionado")
if contratos.filter(F.col("qc_valor_invalido") | F.col("qc_fecha_firma_invalida")
                    | F.col("valor_contrato").isNull() | F.col("fecha_firma").isNull()
                    | (F.year("fecha_firma") != 2024)).limit(1).count():
    raise ValueError("Hay importes o fechas de firma invalidos o fuera de 2024; revisar Silver")

# Codigo de proveedor primero; documento tipado como respaldo; nombre como ultimo recurso.
# La clave es tecnica, no una garantia de anonimato. No exportar datos individuales al repositorio publico.
identidad_proveedor = F.when(F.col("codigo_proveedor").isNotNull(),
    F.concat(F.lit("codigo:"), F.col("codigo_proveedor"))).when(F.col("documento_proveedor").isNotNull(),
    F.concat(F.lit("documento:"), F.coalesce("tipo_documento_proveedor", F.lit("SIN_TIPO")),
             F.lit(":"), F.col("documento_proveedor"))).otherwise(
    F.concat(F.lit("nombre:"), F.upper(F.regexp_replace(
        F.coalesce("proveedor_adjudicado", F.lit("SIN_IDENTIFICACION")), r"\s+", " "))))
contratos = (contratos.withColumn("clave_proveedor", F.sha2(identidad_proveedor, 256))
             .withColumn("mes_firma", F.trunc("fecha_firma", "month"))
             .withColumn("modalidad", F.coalesce(F.upper(F.trim("modalidad_contratacion")), F.lit("SIN INFORMACION")))
             .cache())

Ambiente: dev
Catalogo: project_dev
Raiz de almacenamiento esperada: abfss://lakehouse@addb01.dfs.core.windows.net/


## 3. Indicadores mensuales, proveedores y modalidades
- `kpi_contratacion_mensual`: Una fila por entidad y mes, cantidad, valor contratado y cobertura.
- `kpi_proveedores`: Una fila por entidad y clave de proveedor, sirve para rankings por valor o cantidad.
- `kpi_modalidades`: Una fila por entidad y modalidad contractual.

El valor es el valor contractual publicado, no el gasto ejecutado o pagado. El número de proveedores distintos no es aditivo entre meses. No sumarlo como si fueran proveedores nuevos.

In [0]:
mensual = contratos.groupBy("entidad", "mes_firma").agg(
    F.count(F.lit(1)).alias("numero_contratos"),
    F.sum("valor_contrato").alias("valor_contratado"),
    F.avg("valor_contrato").alias("valor_promedio_contrato"),
    F.countDistinct("clave_proveedor").alias("proveedores_distintos"),
    F.sum(F.col("tiene_proceso").cast("long")).alias("contratos_con_proceso"),
    F.sum((~F.col("tiene_proceso")).cast("long")).alias("contratos_sin_proceso"))
mensual = mensual.withColumn("cobertura_pct", F.round(100.0 * F.col("contratos_con_proceso") / F.col("numero_contratos"), 2))
proveedores = contratos.groupBy("entidad", "clave_proveedor").agg(
    F.count(F.lit(1)).alias("numero_contratos"), F.sum("valor_contrato").alias("valor_contratado"),
    F.sort_array(F.collect_set("proveedor_adjudicado")).alias("nombres_publicados"))
proveedores = proveedores.withColumn("proveedor", F.when(F.size("nombres_publicados") > 0,
    F.element_at("nombres_publicados", 1)).otherwise(F.lit("SIN INFORMACION")))
modalidades = contratos.groupBy("entidad", "modalidad").agg(
    F.count(F.lit(1)).alias("numero_contratos"), F.sum("valor_contrato").alias("valor_contratado"))

## 4. Calidad y reconciliación de totales
- `kpi_calidad_ingesta` conserva la cobertura, variantes de procesos, IDs distintos y repeticiones consolidadas por entidad.
- No coincidencia significa ausencia en esta consulta/snapshot, no incumplimiento ni corrupción.
- Antes de escribir se exige que cantidad y valor por entidad coincidan entre Silver y cada agregado Gold.
Es una barrera contra duplicaciones accidentales en joins y agrupaciones.

In [0]:
calidad = contratos.groupBy("entidad").agg(
    F.count(F.lit(1)).alias("numero_contratos"),
    F.sum(F.col("tiene_proceso").cast("long")).alias("contratos_con_proceso"),
    F.sum((~F.col("tiene_proceso")).cast("long")).alias("contratos_sin_proceso"))
calidad_procesos = procesos.groupBy("entidad").agg(
    F.count(F.lit(1)).alias("variantes_procesos"), F.countDistinct("id_proceso").alias("ids_procesos_distintos"),
    F.sum(F.col("filas_bronze_agrupadas") - 1).alias("duplicados_consolidados"))
calidad = (calidad.join(calidad_procesos, "entidad", "left")
           .withColumn("cobertura_pct", F.round(100.0 * F.col("contratos_con_proceso") / F.col("numero_contratos"), 2)))

totales = contratos.groupBy("entidad").agg(F.count(F.lit(1)).alias("numero_contratos"),
                                          F.sum("valor_contrato").alias("valor_contratado"))
for nombre, agregado in (("mensual", mensual), ("proveedores", proveedores), ("modalidades", modalidades)):
    revision = agregado.groupBy("entidad").agg(F.sum("numero_contratos").alias("numero_contratos"),
                                               F.sum("valor_contratado").alias("valor_contratado"))
    if totales.exceptAll(revision).limit(1).count() or revision.exceptAll(totales).limit(1).count():
        raise ValueError(f"Gold {nombre}: cantidad o valor no reconcilia con Silver")
display(totales.orderBy("entidad"))
display(calidad.orderBy("entidad"))

entidad,numero_contratos,valor_contratado
idartes,3651,129817001887.89
idipron,3233,65451073280.20


entidad,numero_contratos,contratos_con_proceso,contratos_sin_proceso,variantes_procesos,ids_procesos_distintos,duplicados_consolidados,cobertura_pct
idartes,3651,3589,62,3630,3629,37,98.3
idipron,3233,2860,373,2888,2885,0,88.46


## 5. Tablas Delta administradas para el dashboard
Se escriben cuatro tablas en `project_dev.gold` y se comprueba la ubicación fisica en `addb01/lakehouse`.
Todas incluyen `_snapshot_id`, `_anio_cohorte_contratos` y `_run_procesos`. El dashboard debe filtrar el snapshot mostrado para no sumar distintas ingestas.
MERGE actualiza o inserta las claves del snapshot sin borrar otros snapshots ni sobrescribir Silver.

In [0]:
def guardar_gold(df, nombre, dimensiones):
    tabla = f"{esquema}.{nombre}"
    df = (df.withColumn("_snapshot_id", F.lit(snapshot))
          .withColumn("_anio_cohorte_contratos", F.lit(2024))
          .withColumn("_run_procesos", F.lit(run_procesos)))
    claves = ["_snapshot_id"] + dimensiones
    if df.groupBy(*claves).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{tabla}: clave Gold duplicada")
    if spark.catalog.tableExists(tabla):
        detalle = spark.sql(f"DESCRIBE DETAIL {tabla}").first().asDict()
        descripcion = spark.sql(f"DESCRIBE TABLE EXTENDED {tabla}").collect()
        tipo = next((r.data_type for r in descripcion if r.col_name.strip() == "Type"), None)
        if tipo != "MANAGED" or detalle["format"] != "delta" or not detalle["location"].startswith(raiz_destino):
            raise ValueError(f"{tabla}: destino no es Delta administrado en lakehouse de desarrollo")
        if set(df.columns) != set(spark.table(tabla).columns):
            raise ValueError(f"{tabla}: esquema previo incompatible")
        condicion = reduce(lambda a, b: a & b, [F.col(f"t.{c}") == F.col(f"s.{c}") for c in claves])
        (DeltaTable.forName(spark, tabla).alias("t").merge(df.alias("s"), condicion)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    else:
        df.write.format("delta").mode("append").saveAsTable(tabla)
    detalle = spark.sql(f"DESCRIBE DETAIL {tabla}").first().asDict()
    if not detalle["location"].startswith(raiz_destino):
        raise ValueError(f"{tabla}: ubicacion inesperada: {detalle['location']}")
    almacenados = spark.table(tabla).filter(F.col("_snapshot_id") == snapshot)
    if df.select(*claves).exceptAll(almacenados.select(*claves)).limit(1).count() or almacenados.select(*claves).exceptAll(df.select(*claves)).limit(1).count():
        raise ValueError(f"{tabla}: claves persistidas no coinciden con el agregado")
    return {"tabla": tabla, "filas_snapshot": almacenados.count(), "snapshot_id": snapshot,
            "ubicacion_delta": detalle["location"]}

try:
    salida = [
        guardar_gold(mensual, "kpi_contratacion_mensual", ["entidad", "mes_firma"]),
        guardar_gold(proveedores, "kpi_proveedores", ["entidad", "clave_proveedor"]),
        guardar_gold(modalidades, "kpi_modalidades", ["entidad", "modalidad"]),
        guardar_gold(calidad, "kpi_calidad_ingesta", ["entidad"]),
    ]
    display(spark.createDataFrame(salida))
finally:
    contratos.unpersist()

filas_snapshot,snapshot_id,tabla,ubicacion_delta
22,9d02c73832c6df529b662965dc430691a5d39b1da30f11653f13d27545beb1eb,project_dev.gold.kpi_contratacion_mensual,abfss://lakehouse@addb01.dfs.core.windows.net/__unitystorage/catalogs/ab0b04a4-41d3-4906-8c11-9fe611ffcbb4/tables/d8181af1-7715-49fb-bd06-af7e87df895c
4231,9d02c73832c6df529b662965dc430691a5d39b1da30f11653f13d27545beb1eb,project_dev.gold.kpi_proveedores,abfss://lakehouse@addb01.dfs.core.windows.net/__unitystorage/catalogs/ab0b04a4-41d3-4906-8c11-9fe611ffcbb4/tables/4390aebc-6255-42c8-a9e1-e03b56d1cdb4
17,9d02c73832c6df529b662965dc430691a5d39b1da30f11653f13d27545beb1eb,project_dev.gold.kpi_modalidades,abfss://lakehouse@addb01.dfs.core.windows.net/__unitystorage/catalogs/ab0b04a4-41d3-4906-8c11-9fe611ffcbb4/tables/5df6e014-c88c-4077-9a09-5a1c20195dfe
2,9d02c73832c6df529b662965dc430691a5d39b1da30f11653f13d27545beb1eb,project_dev.gold.kpi_calidad_ingesta,abfss://lakehouse@addb01.dfs.core.windows.net/__unitystorage/catalogs/ab0b04a4-41d3-4906-8c11-9fe611ffcbb4/tables/af41b069-caf7-43d8-92d9-acc121074e87


## 6. Evidencias
Conserva las tablas de totales, calidad y ubicaciones. Las evidencias del dashboard pueden mostrar agregados.

In [0]:
dbutils.notebook.exit(json.dumps({"snapshot_id": snapshot, "tablas": salida}, ensure_ascii=False))